<a href="https://colab.research.google.com/github/kaioingz/Relatorio_de_frota_phython/blob/main/Kaio25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IMPORTS PARA CONSTRUÇÃO DO NOSSO AMBIENTE**





**Pandas**: Biblioteca principal para manipulação e análise de dados tabulares.


**Drive**: Conexão necessária para acessar os arquivos Excel salvos no Google Drive.


**OS**: Utilizada para navegar nas pastas e detectar automaticamente os arquivos disponíveis.


**Warnings**: Filtro para ignorar avisos estéticos do Excel, mantendo o console limpo.


**memoria_frota**: Criação do dicionário que servirá como nosso banco de dados em memória.



In [1]:
import pandas as pd
from google.colab import drive
import warnings
import os

**Conexão de Dados**

In [2]:
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Estrutura de Armazenamento Temporário**


Esta seção é o "coração" da performance do nosso script. O objetivo aqui é evitar que o Python precise ler arquivos pesados no Google Drive toda vez que você fizer uma pergunta:

In [3]:
# --- BANCO DE DADOS EM MEMÓRIA ---
# Esta variável guardará os anos já processados para consulta instantânea
if 'memoria_frota' not in locals():
    memoria_frota = {}

**REGRAS DE NEGÓCIO E MOTOR DE CÁLCULO**
***Este bloco define como os dados são agrupados e como a planilha é lida***:

Dicionário de Categorias: Traduz as dezenas de colunas técnicas do DETRAN para grupos compreensíveis (ex: agrupa 'MOTOCICLETA' e 'MOTONETA' em um único total de 'Motocicletas').

Função processar_planilha: É o "cérebro" que executa 4 passos automáticos:

1. Ajuste de Cabeçalho: Testa se os títulos estão na linha 3 ou 4 (evita erros de formatação).

2. Limpeza (Sanitização): Remove acentos, espaços e padroniza tudo para letras maiúsculas.

3. Filtragem Local: Isola apenas os dados da cidade de SÃO PAULO.

4. Consolidação: Soma as colunas específicas de cada categoria e retorna os valores prontos.

In [4]:
# --- DICIONÁRIO DE CATEGORIAS ---
categorias = {
    'Motocicleta': ['CICLOMOTOR', 'MOTONETA', 'MOTOCICLETA', 'SIDE-CAR', 'TRICICLO', 'QUADRICICLO'],
    'Automóvel': ['AUTOMOVEL', 'CAMIONETA', 'CAMINHONETE', 'UTILITARIO'],
    'Ônibus': ['ONIBUS', 'MICRO-ONIBUS'],
    'Caminhão': ['CAMINHAO', 'CAMINHAO TRATOR'],
    'Outros': ['BONDE', 'CHASSI PLATAF', 'REBOQUE', 'SEMI-REBOQUE', 'OUTROS', 'TRATOR ESTEI', 'TRATOR RODAS']
}

def processar_planilha(caminho_arquivo, categorias):
    try:
        df = pd.read_excel(caminho_arquivo, skiprows=2)
        df.columns = df.columns.astype(str).str.strip().str.upper().str.replace('MUNICÍPIO', 'MUNICIPIO').str.replace('MUNICÍPIOS', 'MUNICIPIO')

        if 'MUNICIPIO' not in df.columns:
            df = pd.read_excel(caminho_arquivo, skiprows=3)
            df.columns = df.columns.astype(str).str.strip().str.upper().str.replace('MUNICÍPIO', 'MUNICIPIO').str.replace('MUNICÍPIOS', 'MUNICIPIO')

        if 'MUNICIPIO' not in df.columns: return None

        df['MUNICIPIO'] = df['MUNICIPIO'].astype(str).str.strip().str.upper()
        df_sp = df[df['MUNICIPIO'] == 'SAO PAULO'].copy()

        if df_sp.empty: return None

        totais = {}
        for nome_cat, colunas in categorias.items():
            cols_existentes = [c for c in colunas if c in df_sp.columns]
            totais[nome_cat] = df_sp[cols_existentes].sum(axis=1).values[0] if cols_existentes else 0
        return totais
    except Exception:
        return "ERRO"

**Processamento e Carga em Massa**
**CARGA AUTOMÁTICA DOS DADOS** (ETL)

Este bloco é responsável pelo processo de Extração e Carga:

1. Identifica todos os arquivos .xlsx na pasta do Drive.

2. Extrai o ano de cada arquivo e verifica se ele já foi carregado.

3. Para cada ano novo, percorre os 12 meses, executa a soma das categorias e armazena o resultado consolidado na variável de memória.

**Nota:** Este processo só precisa ser executado uma vez por sessão ou quando novos arquivos forem adicionados à pasta.

In [5]:
caminho_pasta = '/content/drive/MyDrive/onbording/Tabelas_Frota_Geral/'
arquivos = [f for f in os.listdir(caminho_pasta) if f.endswith('.xlsx')]
anos_detectados = sorted(list(set([f.split('_')[0] for f in arquivos])))

meses_nome = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

for ano in anos_detectados:
    if ano not in memoria_frota:
        print(f"-> Carregando ano {ano}...")
        dados_ano = []
        for n in range(1, 13):
            res = processar_planilha(f"{caminho_pasta}{ano}_{n}.xlsx", categorias)
            linha = {'Mês_Num': n, 'Mês': meses_nome[n-1]}
            for cat in categorias.keys():
                linha[cat] = res[cat] if isinstance(res, dict) else 0
            dados_ano.append(linha)
        memoria_frota[ano] = pd.DataFrame(dados_ano)
print("\nCarga Finalizada! Memória pronta.")

-> Carregando ano 2010...
-> Carregando ano 2011...
-> Carregando ano 2012...
-> Carregando ano 2013...
-> Carregando ano 2014...
-> Carregando ano 2015...
-> Carregando ano 2016...
-> Carregando ano 2017...
-> Carregando ano 2018...
-> Carregando ano 2019...
-> Carregando ano 2020...
-> Carregando ano 2021...
-> Carregando ano 2022...
-> Carregando ano 2023...
-> Carregando ano 2024...
-> Carregando ano 2025...

Carga Finalizada! Memória pronta.


**Parâmetros de Busca**
DEFINIÇÃO DA CONSULTA

*ANO_DESEJADO*: O ano que você pretende consultar (deve estar presente na pasta do Drive).

*MES_DESEJADO*: O número do mês (1 a 12). Deixe em branco para gerar o relatório consolidado do ano inteiro.

In [11]:
# Digite aqui o que deseja buscar e execute a célula (Ctrl+Enter)
ANO_DESEJADO = input('Insira o ano desejado: ')
MES_DESEJADO = input('Mês (1-12) ou ENTER para todos: ').strip()

print(f"Alvo definido: {ANO_DESEJADO} | Mês: {MES_DESEJADO if MES_DESEJADO else 'Todos'}")

Insira o ano desejado: 2025
Mês (1-12) ou ENTER para todos: 5
Alvo definido: 2025 | Mês: 5


**Visualização de Resultados**
*EXIBIÇÃO DOS DADOS (DASHBOARD)*

Célula de saída que exibe os dados armazenados na memória. Se um mês foi especificado, exibe a tabela vertical por categoria. Caso contrário, exibe a evolução mensal completa do ano selecionado.

In [12]:
if 'ANO_DESEJADO' in locals() and ANO_DESEJADO in memoria_frota:
    df_mem = memoria_frota[ANO_DESEJADO]

    if MES_DESEJADO != "":
        m_val = int(MES_DESEJADO)
        res = df_mem[df_mem['Mês_Num'] == m_val].drop(columns=['Mês_Num'])
        print(f"\n--- Resultado Individual: {MES_DESEJADO}/{ANO_DESEJADO} ---")
        display(res.melt(id_vars=['Mês'], var_name='Categoria', value_name='Total'))
    else:
        print(f"\n--- Tabela Anual: {ANO_DESEJADO} ---")
        display(df_mem.drop(columns=['Mês_Num']))
else:
    print("Erro: Verifique se o ano foi carregado na Célula 2 ou definido na Célula 3.")


--- Resultado Individual: 5/2025 ---


,Mês,Categoria,Total
0,Mai,Motocicleta,1602752
1,Mai,Automóvel,7852525
2,Mai,Ônibus,94410
3,Mai,Caminhão,187106
4,Mai,Outros,126256
